In [ ]:
def calculate_windel_score(person1_list, person2, person1_dataframe, person2_dataframe):
    """
    person1_list: a list of person_ids to compare to person2
    person2: the person getting the windel score
    person1_dataframe: the whole filtered dataframe that person1_list comes from
    person2_dataframe: the whole filtered dataframe that person2 comes from

    returns the best windel score for person2
    """
    indexes2actuallyplot = []
    jscore_to_bin_dict = {}
    matching_codes = []
    people_compared = {}
    people_bins_avgScores = {}
    averages_with_bins = {}
    split_avg = {}

    for person1 in person1_list:
        person2_df = person2_dataframe[person2_dataframe['person_id'] == person2]
        person1_df = person1_dataframe[person1_dataframe['person_id'] == person1]

        person1_bin_set = set(person1_df['bin'])
        person2_bin_set = set(person2_df['bin'])

        bin_pairs = np.array([(bin1, bin2) for bin1 in person1_bin_set for bin2 in person2_bin_set], dtype=object)

        Asub_list = {bin1: set(person1_df.loc[person1_df['bin'] == bin1, 'concept_code']) for bin1 in person1_bin_set}
        Bsub_list = {bin2: set(person2_df.loc[person2_df['bin'] == bin2, 'concept_code']) for bin2 in person2_bin_set}

        similarity_scores = np.array([
            len(Asub_list[bin1] & Bsub_list[bin2]) / len(Asub_list[bin1] | Bsub_list[bin2]) if len(Asub_list[bin1] | Bsub_list[bin2]) > 0 else 0.0
            for bin1, bin2 in bin_pairs
        ])

        similarity_matrix = similarity_scores.reshape(len(person1_bin_set), len(person2_bin_set))
        mask = np.ones(similarity_matrix.shape, dtype=bool)
        max_dict_ordered = OrderedDict()
        excluded_bins_to_raw_score = OrderedDict()

        while np.any(mask):
            masked_matrix = np.where(mask, similarity_matrix, -np.inf)
            max_index = np.argmax(masked_matrix)
            max_row, max_col = np.unravel_index(max_index, similarity_matrix.shape)
            max_score = similarity_matrix[max_row, max_col]

            tuple_key = tuple(bin_pairs[max_row * len(person2_bin_set) + max_col])
            max_dict_ordered[tuple_key] = max_score
            excluded_bins_to_raw_score.setdefault(tuple_key, []).append(max_score)

            mask[max_row, :] = False
            mask[:, max_col] = False

        sorted_ordered_dict = OrderedDict(sorted(max_dict_ordered.items(), key=lambda x: x[1], reverse=True))
        indexes2actuallyplot.append(list(sorted_ordered_dict.keys()))
        averages_to_actually_graph = [st.mean(sorted_ordered_dict.values())]

        sorted_bins_to_max_score_dict = OrderedDict(sorted(excluded_bins_to_raw_score.items(), key=lambda x: max(x[1]), reverse=True))
        while sorted_bins_to_max_score_dict:
            bins_from_matrix = list(sorted_bins_to_max_score_dict.keys())
            maxes_average = st.mean([val for values in sorted_bins_to_max_score_dict.values() for val in values])
            averages_with_bins.setdefault((person1, person2), []).append({bins_from_matrix[-1]: maxes_average})
            sorted_bins_to_max_score_dict.popitem()

        person1_best_scores = {key[0]: value[0] for key, value in excluded_bins_to_raw_score.items()}
        person2_best_scores = {key[1]: value[0] for key, value in excluded_bins_to_raw_score.items()}

        sorted_by_bins_person1 = OrderedDict(sorted(person1_best_scores.items()))
        sorted_by_bins_person2 = OrderedDict(sorted(person2_best_scores.items()))

        if (person2, person1) not in split_avg:
            split_avg[(person2, person1)] = []

        while sorted_by_bins_person2:
            closest_to_diagnosis_p2 = next(iter(sorted_by_bins_person2))
            curr_average = st.mean(sorted_by_bins_person2.values())
            split_avg[(person2, person1)].append({closest_to_diagnosis_p2: curr_average})
            del sorted_by_bins_person2[closest_to_diagnosis_p2]


    result1 = []
    comparison_ids = [] # This is storing tuples
    # the key is the person, the value is the dictionary of bins to scores
    for key, value in split_avg.items():
        person2_bins = {}
        for val in value:
            for k, v in val.items():
                bin = int(k.split(',')[0])
                person2_bins[bin] = v
        scores = {}
        score_list = []
        curr_score = 0
        for j in bins:
                curr_bin = j
                if j not in person2_bins:
                    curr_score = curr_score
                else:
                    curr_score = person2_bins[j]
                scores[curr_bin] = curr_score
                score_list.append(curr_score)
        max_score = 0
        for item in score_list:
            if item > max_score:
                max_score = item
        if max_score != 1.0:
            result1.append(scores)
            comparison_ids.append(key)


    id_to_list_of_dictionaries = {}
    id_to_list_of_dictionaries[person2] = []
    for group_index in range(len(result1)):
        if comparison_ids[group_index][0] == person2:
            id_to_list_of_dictionaries[person2].append(result1[group_index])

    comparison_functions = {}
    apex_x_values = {}  # Optional: store apex x-values by ID
    total_windel = {}

    for previd, group in id_to_list_of_dictionaries.items():
        total_windel[previd] = []
        i = -1
        for dataset in group:

            i += 1
            x = np.array(list(dataset.keys()))
            y = np.array(list(dataset.values()))

            # Find the cutoff to ignore leading zero values
            index = 0
            while index < len(y) and y[index] == 0:
                index += 1
            y_cutoff = y[index:]
            x_cutoff = x[index:]

            if x_cutoff.size == 0:
                continue

            # Fit a quadratic model
            coefficients = np.polyfit(x_cutoff, y_cutoff, 2)  # coefficients = [a, b, c]
            a, b = coefficients[0], coefficients[1]
            comparison_functions[comparison_ids[i]] = coefficients

            # Create model and predictions
            quadratic_model = np.poly1d(coefficients)
            x_fit = np.linspace(x.min(), x.max(), 100)
            y_fit = quadratic_model(x_fit)

            x_apex = -b / (2 * a)
            apex_x_values[comparison_ids[i]] = x_apex
            total_windel[previd].append(x_apex)

    best_windel = {}
    for identification, w_scores in total_windel.items():
        w_scores = [t for t in w_scores if t < 0]
        best_windel[identification] = max(w_scores)

    return best_windel[person2]

# Example using WINDEL function:

In [ ]:
# first argument is how many days back our bins should go, the second is the size of each bin
# returns two lists, the first is the cuts of each bin and the second is the name of the respective bins
def bins_n_labels(time_range, bin_size):
    bins = []
    labels = []
    n = -time_range
    while n <= 0:
        bins.append(n)
        label = f'{n},'
        n += bin_size
        label += f'{n}'
        labels.append(label)
    return bins, labels[:-1]

#original
# bins, labels = bins_n_labels(15000,100)

#trial 1
bins, labels = bins_n_labels(15000,100)

# arg 1 is how far back you want to look, arg 2 is the size of each bin you want.
print(bins)
# print(bins)
print(len(bins))
# print(labels)
print(len(labels))
# neg_days['bins'] = pd.cut(neg_days['values'], bins=bins, labels=labels)

In [ ]:
#pull in control group people

import os
import subprocess
import numpy as np
import pandas as pd

# This snippet assumes you run setup first

# This code copies file in your Google Bucket and loads it into a dataframe

# Replace 'test.csv' with THE NAME of the file you're going to download from the bucket (don't delete the quotation marks)
name_of_file_in_bucket = 'control_group_malignant.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp '{my_bucket}/data/{name_of_file_in_bucket}' .")

print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
# save dataframe in a csv file in the same workspace as the notebook
control_dataframe = pd.read_csv(name_of_file_in_bucket)
control_dataframe.head()

print(len(control_dataframe))

In [ ]:
# Bins each event
control_dataframe['bin'] = pd.cut(control_dataframe['day_diff'],labels=labels,bins=bins)

value_counts = control_dataframe['bin'].value_counts().sort_index().reset_index()
num_events = value_counts['count'].tolist()

control_dataframe = control_dataframe[pd.notna(control_dataframe['bin'])]
control_dataframe['concept_code'] = control_dataframe['standard_concept_code']

In [ ]:
import matplotlib.pyplot as plt

counts_df = control_dataframe.groupby('person_id').size().reset_index(name='count')

filtered_counts = counts_df[(counts_df["count"] > 100) & (counts_df["count"] < 1200)]
ids = filtered_counts['person_id']
filtered_control = control_dataframe[control_dataframe['person_id'].isin(ids)]
print('Have at least 100 EHRs:')
print(len(filtered_control["person_id"].unique().tolist()))

In [ ]:
# SETUP

import os
import subprocess
import numpy as np
import pandas as pd


# This snippet assumes you run setup first
# Get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')
# List objects in the bucket
# print(subprocess.check_output(f"gsutil ls -r {my_bucket}", shell=True).decode('utf-8'))
# This code copies file in your Google Bucket and loads it into a dataframe

# Replace 'test.csv' with THE NAME of the file you're going to download from the bucket (don't delete the quotation marks)
name_of_file_in_bucket = 'binned_individuals.csv'

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp '{my_bucket}/data/{name_of_file_in_bucket}' .")

# print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
# save dataframe in a csv file in the same workspace as the notebook
binned_individuals = pd.read_csv(name_of_file_in_bucket)
binned_individuals.head()

In [ ]:
counts_ind = binned_individuals.groupby('person_id').size().reset_index(name='count')
filtered_counts = counts_ind[(counts_ind["count"] > 100) & (counts_ind["count"] < 1200)]
ids = filtered_counts['person_id']
filtered_binned = binned_individuals[binned_individuals['person_id'].isin(ids)]
print(len(filtered_binned["person_id"].unique().tolist()))

In [ ]:
#filtered_control = pre-vivors
counts_df = control_dataframe.groupby('person_id').size().reset_index(name='count')
filtered_counts = counts_df[(counts_df["count"] > 100) & (counts_df["count"] < 1200)]
pre_ids = filtered_counts['person_id']
filtered_control = control_dataframe[control_dataframe['person_id'].isin(pre_ids)]

In [ ]:
print(len(filtered_binned))

In [ ]:
#unique_people = test group, randos with >100 EHRs
total_people = filtered_binned['person_id'].unique().tolist()
num_individuals = 75
unique_people = filtered_binned['person_id'].sample(num_individuals, replace=False).tolist()
excluded = set(unique_people) | set(pre_ids)
total_people = [person for person in total_people if person not in excluded]

In [ ]:
print(len(filtered_binned))

In [ ]:
#matched_people = randos with >100 EHRs
num_individuals = 75
new_sample = filtered_binned[filtered_binned['person_id'].isin(total_people)]
matched_people = new_sample['person_id'].sample(num_individuals, replace=False).tolist()

In [ ]:
print(list(set(pre_ids) & set(matched_people)))

In [ ]:
filtered_control.head()

In [ ]:
from collections import OrderedDict
import statistics as st
import numpy as np
import itertools
import copy
import sys
import pandas as pd

fixed_people = filtered_control['person_id'].drop_duplicates().tolist()

previvor_windel_scores = {}

for person in fixed_people:
    person_best_score = calculate_windel_score(matched_people, person, filtered_binned, filtered_control)
    previvor_windel_scores[person] = person_best_score

print(previvor_windel_scores)

In [ ]:
previvor_scores_df = pd.DataFrame(list(previvor_windel_scores.items()), columns=['id', 'best_score'])

#plot previvors windel scores on box plot
import matplotlib.pyplot as plt

previvor_scores_df['best_score'].plot(kind='box')
plt.ylabel('Days before Cancer Diagnosis')
plt.title('Pre-Vivor Best Windel Scores')
plt.show()

In [ ]:
from collections import OrderedDict
import statistics as st
import numpy as np
import itertools
import copy
import sys
import pandas as pd

test_windel_scores = {}

for test_person in unique_people:
    test_person_best_score = calculate_windel_score(matched_people, test_person, filtered_binned, filtered_binned)
    test_windel_scores[test_person] = test_person_best_score

print(test_windel_scores)

In [ ]:
test_scores_df = pd.DataFrame(list(test_windel_scores.items()), columns=['id', 'best_score'])

#plot test windel scores on box plot
import matplotlib.pyplot as plt

test_scores_df['best_score'].plot(kind='box')
plt.ylabel('Days before Cancer Diagnosis')
plt.title('Test Best Windel Scores')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

test_scores = test_scores_df['best_score'].tolist()
previvor_scores = previvor_scores_df['best_score'].tolist()

# Assuming apex_x_list and test_x_list are lists of numbers (e.g., apex x-values)
data = [previvor_scores, test_scores]

plt.figure(figsize=(10, 12))
plt.boxplot(data, labels=["Pre-vivors", "Test"])
plt.ylabel("Days Before Cancer Diagnosis")
plt.title("Comparison of Best WINDEL Scores")
plt.grid(True)
# plt.savefig("best_windel_comparison.pdf", format='pdf', bbox_inches='tight')
plt.savefig("best_windel_comparison_size100.pdf", format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
from scipy.stats import ttest_ind
t_stat, p_value = ttest_ind(previvor_scores, test_scores)
print(t_stat)
print(p_value)

# Original, Split-up code to find a WINDEL score:

This code uses variables from the pre-vivor comparison and gives scores for multiple people.

In [ ]:
#pre-vivors vs matched
from collections import OrderedDict
import statistics as st
import numpy as np
import itertools
import copy
import sys
import pandas as pd

fixed_people = filtered_control['person_id'].drop_duplicates().tolist()

indexes2actuallyplot = []
jscore_to_bin_dict = {}
matching_codes = []
people_compared = {}
people_bins_avgScores = {}
averages_with_bins = {}
split_avg = {}

for rando1 in fixed_people:
    for rando2 in matched_people:
        person2_df = filtered_control[filtered_control['person_id'] == rando1]
        person1_df = filtered_binned[filtered_binned['person_id'] == rando2]

        person1_bin_set = set(person1_df['bin'])
        person2_bin_set = set(person2_df['bin'])

        bin_pairs = np.array([(bin1, bin2) for bin1 in person1_bin_set for bin2 in person2_bin_set], dtype=object)

        Asub_list = {bin1: set(person1_df.loc[person1_df['bin'] == bin1, 'concept_code']) for bin1 in person1_bin_set}
        Bsub_list = {bin2: set(person2_df.loc[person2_df['bin'] == bin2, 'concept_code']) for bin2 in person2_bin_set}

        similarity_scores = np.array([
            len(Asub_list[bin1] & Bsub_list[bin2]) / len(Asub_list[bin1] | Bsub_list[bin2]) if len(Asub_list[bin1] | Bsub_list[bin2]) > 0 else 0.0
            for bin1, bin2 in bin_pairs
        ])

        similarity_matrix = similarity_scores.reshape(len(person1_bin_set), len(person2_bin_set))
        mask = np.ones(similarity_matrix.shape, dtype=bool)
        max_dict_ordered = OrderedDict()
        excluded_bins_to_raw_score = OrderedDict()

        while np.any(mask):
            masked_matrix = np.where(mask, similarity_matrix, -np.inf)
            max_index = np.argmax(masked_matrix)
            max_row, max_col = np.unravel_index(max_index, similarity_matrix.shape)
            max_score = similarity_matrix[max_row, max_col]

            tuple_key = tuple(bin_pairs[max_row * len(person2_bin_set) + max_col])
            max_dict_ordered[tuple_key] = max_score
            excluded_bins_to_raw_score.setdefault(tuple_key, []).append(max_score)

            mask[max_row, :] = False
            mask[:, max_col] = False

        sorted_ordered_dict = OrderedDict(sorted(max_dict_ordered.items(), key=lambda x: x[1], reverse=True))
        indexes2actuallyplot.append(list(sorted_ordered_dict.keys()))
        averages_to_actually_graph = [st.mean(sorted_ordered_dict.values())]

        sorted_bins_to_max_score_dict = OrderedDict(sorted(excluded_bins_to_raw_score.items(), key=lambda x: max(x[1]), reverse=True))
        while sorted_bins_to_max_score_dict:
            bins_from_matrix = list(sorted_bins_to_max_score_dict.keys())
            maxes_average = st.mean([val for values in sorted_bins_to_max_score_dict.values() for val in values])
            averages_with_bins.setdefault((rando1, rando2), []).append({bins_from_matrix[-1]: maxes_average})
            sorted_bins_to_max_score_dict.popitem()

        person1_best_scores = {key[0]: value[0] for key, value in excluded_bins_to_raw_score.items()}
        person2_best_scores = {key[1]: value[0] for key, value in excluded_bins_to_raw_score.items()}

        sorted_by_bins_person1 = OrderedDict(sorted(person1_best_scores.items()))
        sorted_by_bins_person2 = OrderedDict(sorted(person2_best_scores.items()))

        if (rando2, rando1) not in split_avg:
            split_avg[(rando2, rando1)] = []

        while sorted_by_bins_person2:
            closest_to_diagnosis_p2 = next(iter(sorted_by_bins_person2))
            curr_average = st.mean(sorted_by_bins_person2.values())
            split_avg[(rando2, rando1)].append({closest_to_diagnosis_p2: curr_average})
            del sorted_by_bins_person2[closest_to_diagnosis_p2]

In [ ]:
result1 = []
comparison_ids = [] # This is storing tuples
# the key is the person, the value is the dictionary of bins to scores
for key, value in split_avg.items():
    person2_bins = {}
    for val in value:
        for k, v in val.items():
            bin = int(k.split(',')[0])
            person2_bins[bin] = v
    scores = {}
    score_list = []
    curr_score = 0
    for j in bins:
            curr_bin = j
            if j not in person2_bins:
                curr_score = curr_score
            else:
                curr_score = person2_bins[j]
            scores[curr_bin] = curr_score
            score_list.append(curr_score)
    max_score = 0
    for item in score_list:
        if item > max_score:
            max_score = item
    if max_score != 1.0:
        result1.append(scores)
        comparison_ids.append(key)

In [ ]:
id_to_list_of_dictionaries = {}
for person in fixed_people:
    id_to_list_of_dictionaries[person] = []
    for group_index in range(len(result1)):
        if comparison_ids[group_index][1] == person:
            id_to_list_of_dictionaries[person].append(result1[group_index])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np  # Make sure this is imported

plt.figure(figsize=(20,14))

comparison_functions = {}
apex_x_values = {}  # Optional: store apex x-values by ID
total_windel = {}

for previd, group in id_to_list_of_dictionaries.items():
    total_windel[previd] = []
    i = -1
    for dataset in group:

        i += 1
        x = np.array(list(dataset.keys()))
        y = np.array(list(dataset.values()))

        # Find the cutoff to ignore leading zero values
        index = 0
        while index < len(y) and y[index] == 0:
            index += 1
        y_cutoff = y[index:]
        x_cutoff = x[index:]

        if x_cutoff.size == 0:
            continue

        # Fit a quadratic model
        coefficients = np.polyfit(x_cutoff, y_cutoff, 2)  # coefficients = [a, b, c]
        a, b = coefficients[0], coefficients[1]
        comparison_functions[comparison_ids[i]] = coefficients

        # Create model and predictions
        quadratic_model = np.poly1d(coefficients)
        x_fit = np.linspace(x.min(), x.max(), 100)
        y_fit = quadratic_model(x_fit)

        # Only plot lines with a positive quadratic coefficient
#         if a < -5.8e-08:
        plt.plot(x_fit, y_fit)

        # Calculate apex x-value: x = -b / (2a)
        x_apex = -b / (2 * a)
        apex_x_values[comparison_ids[i]] = x_apex
        total_windel[previd].append(x_apex)
    #         print(f"Apex for {comparison_ids[i]}: x = {x_apex:.2f}")

plt.gca().invert_xaxis()
plt.xlabel('Days Before Cancer Diagnosis (Bins)', fontsize=20)
plt.ylabel('Jaccard Score', fontsize=20)
plt.title('Pre-vivors vs Matched', fontsize=25)
plt.xlim(0, -3000)
plt.ylim(-.05, .25)
plt.grid(True)
plt.show()

In [ ]:
best_windel = {}

for identification, w_scores in total_windel.items():
    w_scores = [t for t in w_scores if t < 0]
    best_windel[identification] = max(w_scores)

df = pd.DataFrame(list(best_windel.items()), columns=['id', 'best_score'])
print(df)

In [ ]:
#plot previvors windel scores on box plot
import matplotlib.pyplot as plt

df['best_score'].plot(kind='box')
plt.ylabel('Days before Cancer Diagnosis')
plt.title('Pre-Vivor Best Windel Scores')
plt.show()